# Lab 06 — Mini planner: optimize new sites

Greedy site placement to maximize population served (HTZ automated site planning simplified).

In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from broadcast_planner.core.regions import DEFAULT_REGION, EXAMPLES_DIR
from broadcast_planner.core.grids import read_raster, make_grid
from broadcast_planner.core.sites import load_sites
from broadcast_planner.dvb_t.models import DvbTConfig
from broadcast_planner.planning.optimizer import greedy_add_sites

# 1. Load native 250m matrix rasters
dem, _ = read_raster(DEFAULT_REGION.dem_path())
clutter, _ = read_raster(DEFAULT_REGION.clutter_path())
population, _ = read_raster(DEFAULT_REGION.population_path())
grid = make_grid(DEFAULT_REGION, resolution_m=250.0)

# 2. Load the initial baseline sites
sites = load_sites(EXAMPLES_DIR / 'sites.yaml')

# 3. Configure a highly resilient DVB-T link budget baseline
custom_config = DvbTConfig(
    modulation="16QAM_1/2",       # Lower required C/N threshold for decoding
    guard_interval="1/4",         # Maximize the echo protection window (224 us)
    receiver_mode="fixed_roof"    # Standard rooftop high-gain antenna planning
)

# 4. Run the upgraded optimization engine
print("Starting macro network optimization loop...")
opt = greedy_add_sites(
    region=DEFAULT_REGION,
    grid=grid,
    dem=dem,
    clutter=clutter,
    population=population,
    existing=sites,
    n_add=5,                      # Increased build budget from 3 to 5
    dvb_config=custom_config,     # Injecting optimized link parameters
    topology="HPHT"               # Upgrade hardware from MPMT to macro HPHT masts
)

# 5. Output rich telemetry
print("\n================ METRICS ================")
print(f"Final Optimized Coverage: {opt.final_coverage_percent}%")
print(f"Incremental Population Gains per Step: {opt.incremental_coverage}")
print("=========================================")

for s in opt.selected_sites:
    if s.id.startswith('OPT'):
        print(f"Deployed [Tower {s.id}] -> Coordinates EPSG:3035: ({s.x}, {s.y}) | ERP: {s.erp_kw} kW")

Starting macro network optimization loop...

================ METRICS ================
Final Optimized Coverage: 81.88%
Incremental Population Gains per Step: [2933905.6021957397, 2298925.424556732, 1078870.479133606, 746543.9140205383, 694996.2183532715]
Deployed [Tower OPT_1] -> Coordinates EPSG:3035: (4765000.0, 2325000.0) | ERP: 100.0 kW
Deployed [Tower OPT_2] -> Coordinates EPSG:3035: (4795000.0, 2305000.0) | ERP: 100.0 kW
Deployed [Tower OPT_3] -> Coordinates EPSG:3035: (4785000.0, 2275000.0) | ERP: 100.0 kW
Deployed [Tower OPT_4] -> Coordinates EPSG:3035: (4735000.0, 2345000.0) | ERP: 100.0 kW
Deployed [Tower OPT_5] -> Coordinates EPSG:3035: (4735000.0, 2305000.0) | ERP: 100.0 kW


Launch the Streamlit UI for interactive maps:

```bash
streamlit run app/streamlit_app.py
```